# Practical 5B — Intel Image Classification VGG16

Extracted from `5B.pdf`. Original displayed outputs omitted; executable Python code retained.

**Required local inputs:** `vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5`

In [5]:
# a. Set up the environment
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
# b. Loading and preprocessing data
# Dataset folder path
data_dir = 'LP_IV_Updated Datasets/LP_IV_Updated Datasets/Object Detection/Intel Image Classification'
# Load training data
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(64, 64),
    batch_size=32
)
# Load validation data
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(64, 64),
    batch_size=32
)
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)
# Preprocess images for VGG16
preprocess = tf.keras.applications.vgg16.preprocess_input
train_data = train_ds.map(
    lambda x, y: (preprocess(x), y)
)
val_data = val_ds.map(
    lambda x, y: (preprocess(x), y)
)
# c. Choose the pre-trained VGG16 model
base_model = VGG16(
    weights=None,
    include_top=False,
    input_shape=(64, 64, 3)
)
# Load downloaded VGG16 weights
base_model.load_weights(
    'LP_IV_Updated Datasets/LP_IV_Updated Datasets/Object Detection/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5'
)
# d. Freeze base layers and add custom layers
base_model.trainable = False
inputs = Input(shape=(64, 64, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax')(x)
model = Model(inputs, outputs)
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()
# e. Train the model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)
# f. Compute loss
loss, accuracy = model.evaluate(val_data)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy * 100, "%")
# g. Evaluate the model
print("Model evaluation completed!")
# h. Visualize the predictions
for images, labels in val_ds.take(1):
    predictions = model.predict(
        preprocess(images), verbose=0
    )
    for i in range(3):
        predicted_class = np.argmax(predictions[i])
        actual_class = labels[i].numpy()
        plt.figure(figsize=(4, 4))
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(
            "Predicted: " + class_names[predicted_class] +
            "\nActual: " + class_names[actual_class]
        )
        plt.axis('off')
        plt.show()
# i. Fine-tune the model
# Unfreeze VGG16 base model
base_model.trainable = True
# Freeze all layers except last 4
for layer in base_model.layers[:-4]:
    layer.trainable = False
# Compile with a small learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
# Train again for fine-tuning
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)
# Final evaluation
final_loss, final_accuracy = model.evaluate(val_data)
print("Final Loss:", final_loss)
print("Final Accuracy:", final_accuracy * 100, "%")
# Save the model
model.save('intel_vgg16_model.keras')
print("Model saved successfully!")


Found 12632 files belonging to 6 classes.
Using 10106 files for training.
Found 12632 files belonging to 6 classes.
Using 2526 files for validation.
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Number of classes: 6


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)           │ (None, 64, 64, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ vgg16 (Functional)                   │ (None, 2, 2, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_3           │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 128)                 │          65,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 6)                   │             774 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,781,126 (56.39 MB)

 Trainable params: 66,438 (259.52 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/5
 10/316 ━━━━━━━━━━━━━━━━━━━━ 1:56 381ms/step - accuracy: 0.2469 - loss: 11.0840

KeyboardInterrupt: 